<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/derivative_warp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp

wp.init()
device = "cuda"

Warp not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 MB 7.7 MB/s eta 0:00:00
Warp 1.11.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.1


In [3]:
import warp as wp
import numpy as np

# 1. Initialize Warp (Sets up the CUDA context)
wp.init()
device = "cuda"

@wp.kernel
def central_diff_kernel(f: wp.array(dtype=wp.float32),
                        df: wp.array(dtype=wp.float32),
                        dx: wp.float32,
                        n: wp.int32):

    # Get the unique thread index
    tid = wp.tid()

    if tid < n:
        # LEFT BOUNDARY: Forward Difference
        if tid == 0:
            df[tid] = (f[tid + 1] - f[tid]) / dx

        # RIGHT BOUNDARY: Backward Difference
        elif tid == n - 1:
            df[tid] = (f[tid] - f[tid - 1]) / dx

        # INTERIOR: Central Difference
        else:
            # (f[i+1] - f[i-1]) / 2dx
            df[tid] = (f[tid + 1] - f[tid - 1]) / (2.0 * dx)

# --- Execution ---
N = 2**16
dx = 1.0 / float(N)

# Initialize data using Warp's own memory allocation
# This lives on the GPU immediately
x_np = np.linspace(0.0, 1.0, N, dtype=np.float32)
f_np = np.sin(2.0 * np.pi * x_np).astype(np.float32)

# Move to Warp Arrays (GPU)
f_wp = wp.array(f_np,dtype=wp.float32,  device=device)
df_wp = wp.zeros(n=N, dtype=wp.float32, device=device)

# Launch the kernel: Warp automatically calculates Grid/Block dimensions
wp.launch(kernel=central_diff_kernel,
          dim=N,
          inputs=[f_wp, df_wp, dx, N],
          device=device)

# Synchronize and copy back if needed
wp.synchronize()
df_final = df_wp.numpy()

print(f"Warp derivative at center: {df_final[N//2]}")

Warp derivative at center: -6.28125


In [6]:
@wp.kernel
def central_diff_tile(f: wp.array(dtype=wp.float32),
                      df: wp.array(dtype=wp.float32)):

    # NEW 1.12 FEATURE: Define a tile of 256 elements
    # Warp handles the 'shared memory' and 'halos' behind the scenes
    t = wp.tile_load(f, shape=256, halo=1)

    # Perform math on the tile as if it were a local array
    # (Simplified syntax for central difference)
    res = (t[1:] - t[:-1]) / dx

    wp.tile_store(df, res)

# --- Execution ---
N = 2**16
dx = 1.0 / float(N)

# Initialize data using Warp's own memory allocation
# This lives on the GPU immediately
x_np = np.linspace(0.0, 1.0, N, dtype=np.float32)
f_np = np.sin(2.0 * np.pi * x_np).astype(np.float32)

# Move to Warp Arrays (GPU)
f_wp = wp.array(f_np,dtype=wp.float32,  device=device)
df_wp = wp.zeros(n=N, dtype=wp.float32, device=device)

# Launch the kernel: Warp automatically calculates Grid/Block dimensions
wp.launch(kernel=central_diff_kernel,
          dim=N,
          inputs=[f_wp, df_wp, dx, N],
          device=device)

# Synchronize and copy back if needed
wp.synchronize()
df_final = df_wp.numpy()

print(f"Warp derivative at center: {df_final[N//2]}")

Module __main__ 3211c6d load on device 'cuda:0' took 0.36 ms  (error)


WarpCodegenError: Error while parsing function "central_diff_tile" at /tmp/ipython-input-2755/2492041604.py:7:
    t = wp.tile_load(f, shape=256, halo=1)
;Couldn't find function overload for 'tile_load' that matched inputs with types: [array(ndim=1, dtype=float32), int32, int32]

# 2D array

In [4]:
import warp as wp
import numpy as np

wp.init()
device = "cuda"

@wp.kernel
def central_diff_2d_x(f: wp.array2d(dtype=wp.float32),
                      dfdx: wp.array2d(dtype=wp.float32),
                      dx: wp.float32):

    # Get 2D indices (i = row, j = column)
    i, j = wp.tid()

    # Get grid dimensions
    rows = f.shape[0]
    cols = f.shape[1]

    # Boundary Condition Logic for the X-direction (columns 'j')

    # LEFT BOUNDARY (j=0): Forward Difference
    if j == 0:
        dfdx[i, j] = (f[i, j + 1] - f[i, j]) / dx

    # RIGHT BOUNDARY (j=cols-1): Backward Difference
    elif j == cols - 1:
        dfdx[i, j] = (f[i, j] - f[i, j - 1]) / dx

    # INTERIOR: Central Difference
    else:
        dfdx[i, j] = (f[i, j + 1] - f[i, j - 1]) / (2.0 * dx)

# --- Simulation Setup ---
N = 1024
dx = 1.0 / float(N)

# Initialize a 2D Gaussian or Sine wave on CPU
x = np.linspace(0, 1, N, dtype=np.float32)
y = np.linspace(0, 1, N, dtype=np.float32)
X, Y = np.meshgrid(x, y)
f_np = np.sin(2.0 * np.pi * X) * np.sin(2.0 * np.pi * Y)

# Move to Warp 2D Arrays (GPU)
f_wp = wp.array(f_np,dtype=wp.float32,  device=device)
dfdx_wp = wp.zeros(shape=(N, N), dtype=wp.float32, device=device)

# Launch: The 'dim' is now a tuple (rows, cols)
wp.launch(kernel=central_diff_2d_x,
          dim=(N, N),
          inputs=[f_wp, dfdx_wp, dx],
          device=device)

wp.synchronize()
dfdx_final = dfdx_wp.numpy()

print(f"Derivative calculated for {N}x{N} grid.")

Module __main__ 09e376e load on device 'cuda:0' took 341.88 ms  (compiled)
Derivative calculated for 1024x1024 grid.


# with tile

In [4]:
import warp as wp

# 1. Initialize for the latest architecture
wp.init()
device = "cuda"

# Define Tile Size (e.g., 16x16)
TILE_DIM = 16

@wp.kernel
def central_diff_2d_tile(f: wp.array2d(dtype=wp.float32),
                         dfdx: wp.array2d(dtype=wp.float32),
                         dx: wp.float32):

    # NEW 1.12 API: Load a 2D tile with a halo of 1 in all directions
    # Warp automatically handles the Shared Memory and Syncing
    t = wp.tile_load(f, shape=(TILE_DIM, TILE_DIM), halo=1)

    # Get local coordinates within the tile
    li, lj = wp.tile_tid()

    # 2. Interior Math: Central Difference in X (columns 'j')
    # Using 't' here is ~100x faster than accessing 'f' directly
    # 'lj + 1' is the center, 'lj' is left, 'lj + 2' is right
    val_left  = t[li + 1, lj]
    val_right = t[li + 1, lj + 2]

    # 3. Handle Boundary Logic at the Grid Level
    # (The tile API still allows access to global 'wp.tid()')
    i, j = wp.tid()
    cols = f.shape[1]

    if j == 0:
        # Forward Diff
        dfdx[i, j] = (t[li + 1, lj + 2] - t[li + 1, lj + 1]) / dx
    elif j == cols - 1:
        # Backward Diff
        dfdx[i, j] = (t[li + 1, lj + 1] - t[li + 1, lj]) / dx
    else:
        # Central Diff
        dfdx[i, j] = (val_right - val_left) / (2.0 * dx)

# --- Launch Logic ---
# Warp 1.12 optimizes the launch based on tile shape
wp.launch(kernel=central_diff_2d_tile,
          dim=(1024, 1024),
          inputs=[f_wp, dfdx_wp, 1.0/1024.0],
          device=device)

NameError: name 'dfdx_wp' is not defined